# 39 · 纠错、改写与语义漂移

单元：S06。先修：38。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)；[Manning等《Introduction to Information Retrieval》](https://nlp.stanford.edu/IR-book/)（补充原始教材；章节1、2、6、8）。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：人工拼写、同义词与约束例；非生成模型效果评估。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 安全规则
固定映射battrey→battery、cellphone→phone。原查询和改写同时留存；必须保持实体、否定和约束。

In [2]:
for q in ['battrey repair','cellphone repair official']:
    print(q,'→',rewrite(q))
assert 'official' in rewrite('cellphone repair official')

battrey repair → battery repair
cellphone repair official → phone repair official


## 原查询对照
人工gold为a/c/d。仅改拼写，报告集合漏召与误召；分数不能跨query直接比较。

In [3]:
gold={'a','c','d'}
q='battrey'
for name,x in [('original',q),('corrected',rewrite(q))]:
    ids={d for d,s in field_rank(x)}
    print(name,sorted(ids),'miss',sorted(gold-ids),'false',sorted(ids-gold))

original [] miss ['a', 'c', 'd'] false []
corrected ['a', 'b', 'c', 'd'] miss [] false ['b']


## 失败改写
故意移除official并扩fruit造成漂移。只改变aggressive；检查约束保留，而非看语言是否流畅。

In [4]:
q='apple repair official'
good=rewrite(q); bad=rewrite(q,aggressive=True)
print(good,field_rank(good,required=['official']))
print(bad,field_rank(bad))
assert 'official' not in bad

apple repair official [('a', 5.0)]
apple repair fruit [('b', 6.0), ('d', 6.0), ('a', 4.0), ('c', 3.0)]


## 回退
约束验证失败回原查询；新增改写也可能无结果。实际线上还要评估频次、人工判断与延迟。

In [5]:
def guarded(q):
    x=rewrite(q,aggressive=True)
    return q if ('official' in terms(q) and 'official' not in terms(x)) else x
assert guarded(q)==q
print('fallback',guarded(q))

fallback apple repair official


## 本次保存结果的解读

拼写battrey原查询没有候选，纠正battery后能找回文档，但spam也能匹配，故纠错不保证精度。激进改写删除official并添加fruit，约束检查选择回原查询。收益、误召与约束保留须一起检查。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。